# Notebook 04: Model Experiments & Chronological Validation
**Retail Demand Forecasting & Inventory Intelligence Platform**

Compares Naive, Seasonal Naive, Moving Average baselines against Random Forest and XGBoost with out-of-time chronological splitting and retail metrics (WAPE, MAE, RMSE, Bias).

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
from src.config import settings
from src.models.baselines import BaselineForecaster
from src.models.forecaster import RetailDemandForecaster
from src.models.evaluation import calculate_metrics

features_df = pd.read_parquet(settings.FEATURES_DATA_DIR / 'features.parquet')
features_df['date'] = pd.to_datetime(features_df['date'])
cutoff = features_df['date'].max() - pd.Timedelta(days=60)

train_df = features_df[features_df['date'] < cutoff]
test_df = features_df[features_df['date'] >= cutoff].copy()

# Evaluate Baselines
baselines = BaselineForecaster.evaluate_baselines(test_df)
print("Baseline Results:", baselines)

# Load Trained XGBoost Model
forecaster = RetailDemandForecaster.load(settings.MODEL_ARTIFACT_PATH)
preds = forecaster.predict(test_df)
ml_metrics = calculate_metrics(test_df['sales'].values, preds)
print("XGBoost Metrics:", ml_metrics)